# Retail Intelligence Pipeline — EDA
**Dataset:** BigMart Sales Data (Kaggle) — `Train.csv`  
**Shape esperado:** ~8,523 filas × 12 columnas (incluye `Item_Outlet_Sales`)  
**Objetivo:** Entender estructura, calidad de datos y distribuciones clave antes de ingestar a Supabase.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

## 1. Carga del dataset

In [2]:

df = pd.read_csv('../data/raw/blinkit_grocery_data.csv')
print(f'Shape: {df.shape}')
df.head()

Shape: (5681, 11)


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type
0,FDW58,20.750,Low Fat,0.007565,Snack Foods,107.8622,OUT049,1999,Medium,Tier 1,Supermarket Type1
1,FDW14,8.300,reg,0.038428,Dairy,87.3198,OUT017,2007,NaN,Tier 2,Supermarket Type1
2,NCN55,14.600,Low Fat,0.099575,Others,241.7538,OUT010,1998,NaN,Tier 3,Grocery Store
3,FDQ58,7.315,Low Fat,0.015388,Snack Foods,155.0340,OUT017,2007,NaN,Tier 2,Supermarket Type1
4,FDY38,NaN,Regular,0.118599,Dairy,234.2300,OUT027,1985,Medium,Tier 3,Supermarket Type3


## 2. Estructura y tipos de datos

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 3. Calidad de datos — Nulos y duplicados

In [ ]:
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(2)
quality = pd.DataFrame({'nulls': nulls, 'pct': null_pct}).query('nulls > 0')
print(quality)
# Esperado:
# Item_Weight    ~17%  -> imputar con mediana por Item_Type
# Outlet_Size    ~28%  -> imputar con moda por Outlet_Type

In [ ]:
print(f'Duplicados: {df.duplicated().sum()}')

# Item_Fat_Content tiene 5 valores que son realmente 2
print('\nItem_Fat_Content antes de normalizar:')
print(df['Item_Fat_Content'].value_counts())

In [ ]:
# Item_Visibility = 0 es un error (producto invisible no tiene sentido)
print(f'Registros con Item_Visibility = 0: {(df["Item_Visibility"] == 0).sum()}')

## 4. Distribución de ventas (Item_Outlet_Sales)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Item_Outlet_Sales'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de Ventas por Item-Outlet')
axes[0].set_xlabel('Ventas ($)')
axes[0].set_ylabel('Frecuencia')

axes[1].hist(np.log1p(df['Item_Outlet_Sales']), bins=40, color='coral', edgecolor='white')
axes[1].set_title('Distribución log(Ventas + 1) — más normal')
axes[1].set_xlabel('log(Ventas)')

plt.tight_layout()
plt.savefig('../data/processed/dist_ventas.png', dpi=150)
plt.show()

print(f'Media: ${df["Item_Outlet_Sales"].mean():.0f}')
print(f'Mediana: ${df["Item_Outlet_Sales"].median():.0f}')
print(f'Skewness: {df["Item_Outlet_Sales"].skew():.2f}')

## 5. Ventas por categoría de producto

In [ ]:
ventas_categoria = (
    df.groupby('Item_Type')['Item_Outlet_Sales']
    .agg(['sum', 'mean', 'count'])
    .rename(columns={'sum': 'total_sales', 'mean': 'avg_sales', 'count': 'n_items'})
    .sort_values('total_sales', ascending=False)
)
print(ventas_categoria)

fig, ax = plt.subplots(figsize=(12, 6))
ventas_categoria['total_sales'].plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Total de Ventas por Categoría')
ax.set_xlabel('Categoría')
ax.set_ylabel('Ventas totales ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../data/processed/ventas_por_categoria.png', dpi=150)
plt.show()

## 6. Ventas por tipo y tamaño de outlet

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.groupby('Outlet_Type')['Item_Outlet_Sales'].mean().sort_values().plot(
    kind='barh', ax=axes[0], color='teal'
)
axes[0].set_title('Venta Promedio por Tipo de Outlet')
axes[0].set_xlabel('Ventas promedio ($)')

df.groupby('Outlet_Size')['Item_Outlet_Sales'].mean().sort_values().plot(
    kind='barh', ax=axes[1], color='darkorange'
)
axes[1].set_title('Venta Promedio por Tamaño de Outlet')
axes[1].set_xlabel('Ventas promedio ($)')

plt.tight_layout()
plt.savefig('../data/processed/ventas_por_outlet.png', dpi=150)
plt.show()

## 7. Correlación entre variables numéricas

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de Correlación')
plt.tight_layout()
plt.savefig('../data/processed/correlacion.png', dpi=150)
plt.show()

# Correlación con el target
print('\nCorrelación con Item_Outlet_Sales:')
print(corr['Item_Outlet_Sales'].sort_values(ascending=False))

## 8. Insights clave (del análisis del archivo de features)

Los siguientes insights se derivan del análisis previo del dataset de features. Completar con hallazgos del target una vez disponible.

### Calidad de datos
1. **`Item_Fat_Content`** tiene 5 valores que son realmente 2: `LF`/`low fat` → `Low Fat`; `reg` → `Regular`. Limpiar antes de ingestar.
2. **`Item_Weight`** tiene 17.18% nulos → imputar con **mediana por `Item_Type`** (productos similares pesan similar).
3. **`Outlet_Size`** tiene 28.27% nulos → imputar con **moda por `Outlet_Type`** (el cruce es determinístico: Grocery Store → Small, Type3 → Medium).
4. **`Item_Visibility` = 0** en 353 registros (6.2%) → error de datos. Imputar con media por `Item_Type`.

### Negocio
5. **Supermarket Type1 domina** (65.4% de registros) — es el canal principal.
6. **Expansión en ciudades intermedias**: Tier 3 tiene el 39.3% de los outlets vs 28% en Tier 1.
7. **El outlet de 1985** tiene el doble de SKUs (~976) que cualquier otro — es el outlet flagship de la red.
8. **`Item_MRP`** distribuido uniformemente en cuartiles → el portafolio cubre todos los segmentos de precio por diseño.